# GNN Laundering Inference

This notebook loads the saved **GraphSAGE A+B (Synergy)** model and provides two main functions:

1. **`run_inference_full_dataset()`** — Preprocesses raw CSV data (or loads pre-computed features), builds the graph, and runs the GNN on all nodes. Returns risk scores for every account.
2. **`get_laundering_prediction(account_id)`** — Looks up a specific account's risk score and binary laundering flag from the cached full-graph inference.

> **Why full-graph inference is required:** GNNs use message passing — each node's prediction depends on its neighbors' features. You cannot predict a single node in isolation. The workflow is always: build full graph → run model on all nodes → look up specific accounts.

**Data files used:**
- Raw CSV: `datasets/ibm_aml/HI-Small_Trans.csv`
- Pre-computed features (optional, speeds up): `backend/data/kaggle/working/processed_data/`
- Model weights: `backend/model/run_1_GraphSAGE_A+B_(Synergy).pkl`

In [28]:
# ============================================
# Imports & Paths
# ============================================
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler
from scipy import sparse

# --- Paths (resolve repo root by walking up until we find backend/model) ---
PROJECT_ROOT = Path.cwd()
for _candidate in [PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT / "GenAI-Genesis", PROJECT_ROOT.parent / "GenAI-Genesis"]:
    if (_candidate / "backend" / "model").exists():
        PROJECT_ROOT = _candidate
        break

MODEL_PATH = PROJECT_ROOT / "backend" / "model" / "run_1_GraphSAGE_A+B_(Synergy).pkl"
RAW_CSV_PATH = PROJECT_ROOT / "datasets" / "ibm_aml" / "HI-Small_Trans.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "backend" / "data" / "kaggle" / "working" / "processed_data"
FEATURE_DIR = PROCESSED_DATA_DIR / "features"
META_DIR = PROCESSED_DATA_DIR / "metadata"

DEFAULT_RISK_THRESHOLD = 0.5

print(f"Project root:          {PROJECT_ROOT}")
print(f"Model exists:          {MODEL_PATH.exists()}  ({MODEL_PATH})")
print(f"Raw CSV exists:        {RAW_CSV_PATH.exists()}  ({RAW_CSV_PATH})")
print(f"Pre-computed features: {PROCESSED_DATA_DIR.exists()}  ({PROCESSED_DATA_DIR})")

Project root:          c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\GenAI-Genesis
Model exists:          True  (c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\GenAI-Genesis\backend\model\run_1_GraphSAGE_A+B_(Synergy).pkl)
Raw CSV exists:        True  (c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\GenAI-Genesis\datasets\ibm_aml\HI-Small_Trans.csv)
Pre-computed features: True  (c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\GenAI-Genesis\backend\data\kaggle\working\processed_data)


## Model Architecture

GraphSAGE with residual connections and batch normalization (must match the saved weights).

In [29]:
# ============================================
# GraphSAGE Model Architecture (must match training code)
# ============================================

class GraphSAGE_AML(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=3, dropout=0.3, aggr="mean", num_classes=2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(SAGEConv(input_dim, hidden_dim, aggr=aggr))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim, aggr=aggr))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.cls = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x_prev = x
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            if x.shape == x_prev.shape:
                x = x + x_prev  # residual connection
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.cls(x)
        return F.log_softmax(x, dim=1)

print("GraphSAGE_AML defined.")

GraphSAGE_AML defined.


## Preprocessing Pipeline

These functions replicate the exact preprocessing from `gnn_fraud_detection_iis_project_amin.ipynb`:
1. **Load & clean CSV** — rename columns, parse timestamps, extract time features
2. **Build graph** — accounts as nodes, transactions as directed edges
3. **Feature Block A (Behavioral)** — 54 features: in/out transaction stats, flow metrics, log-scaled heavy-tail features, cyclical time encodings, payment format entropy
4. **Feature Block B (Random Walk)** — 4 features: return rate, return frequency, first return step, unique node ratio
5. **StandardScaler normalization** — fit on train split only (no data leakage)

In [30]:
# ============================================
# Preprocessing: CSV loading & graph construction
# ============================================

def load_and_clean_csv(csv_path):
    """Load IBM AML CSV, rename columns, parse timestamps, add time features."""
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"Account": "From Account", "Account.1": "To Account"})
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["Timestamp"]).reset_index(drop=True)
    df["ts"] = (df["Timestamp"].astype("int64") // 10**9).astype(np.int64)
    df["Amount Paid"] = pd.to_numeric(df["Amount Paid"], errors="coerce").fillna(0.0)
    df["Amount Received"] = pd.to_numeric(df["Amount Received"], errors="coerce").fillna(0.0)
    df["date"] = df["Timestamp"].dt.date
    df["hour"] = df["Timestamp"].dt.hour.astype(np.int16)
    df["weekday"] = df["Timestamp"].dt.dayofweek.astype(np.int16)
    return df


def build_graph(df):
    """Build node/edge mappings and PyG-compatible edge_index from transaction DataFrame."""
    all_accounts = pd.Index(
        pd.concat([df["From Account"], df["To Account"]], ignore_index=True).unique()
    )
    account_to_id = {acc: i for i, acc in enumerate(all_accounts)}
    id_to_account = {i: acc for acc, i in account_to_id.items()}
    N = len(account_to_id)

    src = df["From Account"].map(account_to_id).to_numpy(dtype=np.int64)
    dst = df["To Account"].map(account_to_id).to_numpy(dtype=np.int64)
    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)

    # Node labels: any account involved in a laundering transaction gets label 1
    laundering_df = df[df["Is Laundering"] == 1]
    laundering_accounts = pd.Index(
        pd.concat([laundering_df["From Account"], laundering_df["To Account"]], ignore_index=True).unique()
    )
    y = torch.zeros(N, dtype=torch.long)
    laundering_ids = [account_to_id[a] for a in laundering_accounts if a in account_to_id]
    y[laundering_ids] = 1

    return account_to_id, id_to_account, edge_index, y, N


def make_temporal_splits(df, account_to_id):
    """70/15/15 temporal split. Returns cumulative edge indices and node presence masks."""
    N = len(account_to_id)
    i_train_end = int(0.70 * len(df))
    i_val_end = int(0.85 * len(df))

    def build_ei(sub_df):
        s = sub_df["From Account"].map(account_to_id).to_numpy(dtype=np.int64)
        d = sub_df["To Account"].map(account_to_id).to_numpy(dtype=np.int64)
        return torch.tensor(np.vstack([s, d]), dtype=torch.long)

    def node_presence(sub_df):
        present = np.zeros(N, dtype=bool)
        present[sub_df["From Account"].map(account_to_id).to_numpy(np.int64)] = True
        present[sub_df["To Account"].map(account_to_id).to_numpy(np.int64)] = True
        return torch.tensor(present, dtype=torch.bool)

    edge_index_train = build_ei(df.iloc[:i_train_end])
    edge_index_val = build_ei(df.iloc[:i_val_end])
    edge_index_test = build_ei(df)  # full graph for test

    train_mask = node_presence(df.iloc[:i_train_end])
    val_mask = node_presence(df.iloc[:i_val_end])
    test_mask = node_presence(df)

    return {
        "edge_index_train": edge_index_train,
        "edge_index_val": edge_index_val,
        "edge_index_test": edge_index_test,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "i_train_end": i_train_end,
        "i_val_end": i_val_end,
    }

print("CSV loading & graph construction functions ready.")

CSV loading & graph construction functions ready.


In [31]:
# ============================================
# Feature Block A: Behavioral & Temporal Statistics (54 features)
# ============================================

def _payment_entropy(series):
    """Information entropy of payment format distribution."""
    counts = series.value_counts(normalize=True)
    return -(counts * np.log2(counts.clip(lower=1e-12))).sum()


def _make_time_block(df_side, group_col, prefix):
    """Cyclical time encoding (hour/weekday) + concentration metrics for one direction."""
    df_side = df_side.copy()
    df_side["hour_sin"] = np.sin(2 * np.pi * df_side["hour"] / 24.0)
    df_side["hour_cos"] = np.cos(2 * np.pi * df_side["hour"] / 24.0)
    df_side["wday_sin"] = np.sin(2 * np.pi * df_side["weekday"] / 7.0)
    df_side["wday_cos"] = np.cos(2 * np.pi * df_side["weekday"] / 7.0)

    agg = df_side.groupby(group_col, sort=False).agg(
        hour_sin_mean=("hour_sin", "mean"),
        hour_cos_mean=("hour_cos", "mean"),
        wday_sin_mean=("wday_sin", "mean"),
        wday_cos_mean=("wday_cos", "mean"),
    )
    agg[f"{prefix}_hour_conc"] = np.sqrt(agg["hour_sin_mean"]**2 + agg["hour_cos_mean"]**2)
    agg[f"{prefix}_wday_conc"] = np.sqrt(agg["wday_sin_mean"]**2 + agg["wday_cos_mean"]**2)

    return agg[["hour_sin_mean", "hour_cos_mean", "wday_sin_mean", "wday_cos_mean",
                f"{prefix}_hour_conc", f"{prefix}_wday_conc"]].rename(
        columns=lambda c: f"{prefix}_{c}" if not c.startswith(prefix) else c
    )


def build_block_a(df_side, accounts_index):
    """Build 54-dim behavioral feature vector for all accounts."""
    N = len(accounts_index)
    acc_to_idx = {acc: i for i, acc in enumerate(accounts_index)}

    # --- Outbound stats ---
    out_agg = df_side.groupby("From Account", sort=False).agg(
        out_count=("From Account", "size"),
        out_amount_sum=("Amount Paid", "sum"),
        out_amount_mean=("Amount Paid", "mean"),
        out_amount_std=("Amount Paid", "std"),
        out_amount_max=("Amount Paid", "max"),
        out_unique_to=("To Account", "nunique"),
        out_unique_banks=("To Bank", "nunique"),
        out_unique_curr=("Payment Currency", "nunique"),
    )

    # --- Inbound stats ---
    in_agg = df_side.groupby("To Account", sort=False).agg(
        in_count=("To Account", "size"),
        in_amount_sum=("Amount Received", "sum"),
        in_amount_mean=("Amount Received", "mean"),
        in_amount_std=("Amount Received", "std"),
        in_amount_max=("Amount Received", "max"),
        in_unique_from=("From Account", "nunique"),
        in_unique_banks=("From Bank", "nunique"),
        in_unique_curr=("Receiving Currency", "nunique"),
    )

    # Merge into account-level DataFrame
    feat = pd.DataFrame(index=accounts_index).fillna(0)
    for col in out_agg.columns:
        feat[col] = out_agg[col].reindex(accounts_index).fillna(0)
    for col in in_agg.columns:
        feat[col] = in_agg[col].reindex(accounts_index).fillna(0)

    # --- Derived flow metrics ---
    feat["total_count"] = feat["out_count"] + feat["in_count"]
    feat["net_flow"] = feat["in_amount_sum"] - feat["out_amount_sum"]
    feat["count_imbalance"] = feat["in_count"] / (feat["out_count"] + 1.0)
    feat["amount_ratio_in_out"] = (feat["in_amount_sum"] + 1.0) / (feat["out_amount_sum"] + 1.0)
    feat["log_amount_ratio_in_out"] = np.log(feat["amount_ratio_in_out"].astype(np.float32))
    feat["unique_partners"] = feat["out_unique_to"] + feat["in_unique_from"]

    # --- Log-scaled heavy-tail features ---
    heavy = [
        "out_count", "in_count", "total_count", "out_amount_sum", "in_amount_sum",
        "out_amount_mean", "in_amount_mean", "out_amount_std", "in_amount_std",
        "out_amount_max", "in_amount_max", "out_unique_to", "in_unique_from",
        "unique_partners", "out_unique_banks", "in_unique_banks", "out_unique_curr", "in_unique_curr",
    ]
    for c in heavy:
        feat[f"log1p_{c}"] = np.log1p(feat[c].clip(lower=0).astype(np.float32))

    # --- Cyclical time features ---
    out_time = _make_time_block(df_side, "From Account", "out")
    in_time = _make_time_block(df_side, "To Account", "in")
    for col in out_time.columns:
        feat[col] = out_time[col].reindex(accounts_index).fillna(0)
    for col in in_time.columns:
        feat[col] = in_time[col].reindex(accounts_index).fillna(0)

    # --- Payment format entropy ---
    out_entropy = df_side.groupby("From Account")["Payment Format"].apply(_payment_entropy)
    in_entropy = df_side.groupby("To Account")["Payment Format"].apply(_payment_entropy)
    feat["out_payment_format_entropy"] = out_entropy.reindex(accounts_index).fillna(0)
    feat["in_payment_format_entropy"] = in_entropy.reindex(accounts_index).fillna(0)

    # Fill NaN/Inf
    feat = feat.replace([np.inf, -np.inf], 0).fillna(0)
    return feat.values.astype(np.float32)

print("Feature Block A (behavioral, 54-dim) ready.")

Feature Block A (behavioral, 54-dim) ready.


In [32]:
# ============================================
# Feature Block B: Random Walk Dynamics (4 features)
# ============================================

def compute_random_walk_features(edge_index, num_nodes, walk_length=10, num_walks=20, device="cpu"):
    """
    Compute random walk features for all nodes: return rate, return frequency,
    first return mean, unique node ratio. Uses GPU if available.
    """
    ei = edge_index.to(device)
    # Build adjacency list via sparse tensor for fast neighbor lookup
    # Group edges by source node
    src, dst = ei[0], ei[1]
    deg = torch.zeros(num_nodes, dtype=torch.long, device=device)
    deg.scatter_add_(0, src, torch.ones_like(src))

    # For nodes with no outgoing edges, self-loop so walk doesn't break
    # Build CSR-style neighbor list
    sort_idx = torch.argsort(src)
    sorted_dst = dst[sort_idx]
    row_ptr = torch.zeros(num_nodes + 1, dtype=torch.long, device=device)
    row_ptr[1:] = deg.cumsum(0)

    # Pre-allocate results
    rw_return_rate = torch.zeros(num_nodes, device=device)
    rw_return_freq = torch.zeros(num_nodes, device=device)
    rw_first_return = torch.zeros(num_nodes, device=device)
    rw_unique_ratio = torch.zeros(num_nodes, device=device)

    batch_size = 200_000
    for start in range(0, num_nodes, batch_size):
        end = min(start + batch_size, num_nodes)
        batch_nodes = torch.arange(start, end, device=device)
        B = len(batch_nodes)

        # walks shape: (B, num_walks, walk_length+1)
        walks = torch.zeros(B, num_walks, walk_length + 1, dtype=torch.long, device=device)
        walks[:, :, 0] = batch_nodes.unsqueeze(1).expand(-1, num_walks)

        for step in range(walk_length):
            current = walks[:, :, step]  # (B, num_walks)
            flat = current.reshape(-1)
            d = deg[flat]
            # Nodes with degree 0 stay in place (self-loop)
            has_neighbors = d > 0
            rand_idx = (torch.rand(flat.shape, device=device) * d.float().clamp(min=1)).long()
            next_nodes = flat.clone()
            if has_neighbors.any():
                offsets = row_ptr[flat[has_neighbors]] + rand_idx[has_neighbors].clamp(max=d[has_neighbors] - 1)
                next_nodes[has_neighbors] = sorted_dst[offsets]
            walks[:, :, step + 1] = next_nodes.reshape(B, num_walks)

        start_ids = batch_nodes.unsqueeze(1).unsqueeze(2)  # (B, 1, 1)

        # Return rate: fraction of walks that revisit start
        hits = (walks[:, :, 1:] == start_ids).any(dim=2).float()  # (B, num_walks)
        rw_return_rate[start:end] = hits.mean(dim=1)

        # Return frequency: avg number of revisits to start per walk
        hit_counts = (walks[:, :, 1:] == start_ids).sum(dim=2).float()
        rw_return_freq[start:end] = hit_counts.mean(dim=1)

        # First return mean: avg step of first return (0 if never returns)
        big = walk_length + 1
        step_idx = torch.arange(1, walk_length + 1, device=device).unsqueeze(0).unsqueeze(0)
        is_start = (walks[:, :, 1:] == start_ids)  # (B, num_walks, walk_length)
        first_steps = torch.where(is_start, step_idx, torch.tensor(big, device=device)).min(dim=2).values.float()
        returned_mask = hits > 0
        first_steps_masked = torch.where(returned_mask, first_steps, torch.zeros_like(first_steps))
        returned_counts = returned_mask.sum(dim=1).clamp(min=1)
        rw_first_return[start:end] = first_steps_masked.sum(dim=1) / returned_counts

        # Unique node ratio: fraction of distinct nodes visited
        sorted_walks, _ = walks[:, :, 1:].sort(dim=2)
        unique_counts = (sorted_walks[:, :, 1:] != sorted_walks[:, :, :-1]).sum(dim=2).float() + 1
        rw_unique_ratio[start:end] = (unique_counts / walk_length).mean(dim=1)

    features = torch.stack([rw_return_rate, rw_return_freq, rw_first_return, rw_unique_ratio], dim=1)
    return features.cpu()

print("Feature Block B (random walk, 4-dim) ready.")

Feature Block B (random walk, 4-dim) ready.


In [33]:
# ============================================
# Full preprocessing: raw CSV → features + graph (from scratch)
# ============================================

def preprocess_from_csv(csv_path, device="cpu"):
    """
    End-to-end preprocessing: loads CSV, builds graph, computes Block A + B features,
    normalizes with StandardScaler (fit on train split), and returns everything needed
    for inference.

    Returns dict with keys:
        X_test, edge_index_test, y, account_to_id, id_to_account, test_mask, N
    """
    print("Loading CSV...")
    df = load_and_clean_csv(csv_path)
    print(f"  {len(df):,} transactions loaded.")

    print("Building graph...")
    account_to_id, id_to_account, edge_index, y, N = build_graph(df)
    accounts_index = pd.Index([id_to_account[i] for i in range(N)])
    print(f"  {N:,} nodes, {edge_index.shape[1]:,} edges.")

    print("Temporal split (70/15/15)...")
    splits = make_temporal_splits(df, account_to_id)

    # --- Block A: behavioral features (compute on each split's data window) ---
    print("Computing Block A (behavioral features)...")
    i_train_end = splits["i_train_end"]
    i_val_end = splits["i_val_end"]

    X_a_train_raw = build_block_a(df.iloc[:i_train_end], accounts_index)
    X_a_test_raw = build_block_a(df, accounts_index)  # full data for test

    # --- Block B: random walk features (on each split's edge structure) ---
    print("Computing Block B (random walk features)... this may take a few minutes.")
    rw_device = device
    X_b_train_raw = compute_random_walk_features(
        splits["edge_index_train"], N, device=rw_device
    ).numpy()
    X_b_test_raw = compute_random_walk_features(
        splits["edge_index_test"], N, device=rw_device
    ).numpy()

    # --- Standardize (fit on train, transform test) ---
    print("Normalizing features (StandardScaler fit on train)...")
    train_mask_np = splits["train_mask"].numpy()

    scaler_a = StandardScaler()
    scaler_a.fit(X_a_train_raw[train_mask_np])
    X_a_test = scaler_a.transform(X_a_test_raw)

    scaler_b = StandardScaler()
    scaler_b.fit(X_b_train_raw[train_mask_np])
    X_b_test = scaler_b.transform(X_b_test_raw)

    # --- Concatenate A + B ---
    X_test = np.concatenate([X_a_test, X_b_test], axis=1)
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)
    X_test = torch.tensor(X_test, dtype=torch.float32)

    print(f"Final feature matrix: {X_test.shape} (expected dim=58)")
    return {
        "X_test": X_test,
        "edge_index_test": splits["edge_index_test"],
        "y": y,
        "account_to_id": account_to_id,
        "id_to_account": id_to_account,
        "test_mask": splits["test_mask"],
        "N": N,
    }

print("Full preprocessing pipeline ready.")

Full preprocessing pipeline ready.


## Load Model & Data

Loads the saved GraphSAGE weights, then either:
- **Fast path:** loads pre-computed features from `backend/data/kaggle/working/processed_data/` (if available)
- **Slow path:** preprocesses the raw CSV from scratch (computes all features, ~minutes)

In [34]:
# ============================================
# Load model weights
# ============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = GraphSAGE_AML(
    input_dim=int(ckpt["input_dim"]),
    hidden_dim=ckpt["config"]["hidden_dim"],
    num_layers=ckpt["config"]["num_layers"],
    dropout=ckpt["config"]["dropout"],
    aggr=ckpt["config"].get("aggr", "mean"),
)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.to(device)
model.eval()

print(f"Model loaded: GraphSAGE | input_dim={ckpt['input_dim']} | "
      f"hidden={ckpt['config']['hidden_dim']} | layers={ckpt['config']['num_layers']}")

Model loaded: GraphSAGE | input_dim=58 | hidden=64 | layers=3


In [35]:
# ============================================
# Load data: fast path (pre-computed) or slow path (from raw CSV)
# ============================================

def load_precomputed_features(device="cpu"):
    """Load pre-computed features, graph, and account maps from PROCESSED_DATA_DIR."""
    base_data = torch.load(META_DIR / "base_graph_data.pt", map_location=device, weights_only=False)
    edge_index = base_data.edge_index
    y = base_data.y

    with open(META_DIR / "account_maps.pkl", "rb") as f:
        maps = pickle.load(f)
    account_to_id = maps["account_to_id"]
    id_to_account = maps["id_to_account"]

    # Load A+B feature blocks for test split and concatenate
    def load_block(name, split="test"):
        with open(PROCESSED_DATA_DIR / "feature_manifest.json", "r") as f:
            man = json.load(f)
        path = Path(man["feature_blocks"][name]["paths"][split])
        if not path.exists():
            path = FEATURE_DIR / path.name
        return torch.load(path, weights_only=False)

    X_a = load_block("behavioral_A", "test")
    X_b = load_block("random_walk_B", "test")
    X_test = torch.cat([X_a, X_b], dim=1)

    N = base_data.num_nodes
    test_mask = torch.ones(N, dtype=torch.bool)  # all nodes present in full graph

    return {
        "X_test": X_test,
        "edge_index_test": edge_index,
        "y": y,
        "account_to_id": account_to_id,
        "id_to_account": id_to_account,
        "test_mask": test_mask,
        "N": N,
    }


# Try fast path first, fall back to slow path
if PROCESSED_DATA_DIR.exists():
    print("Pre-computed features found — loading (fast path)...")
    data = load_precomputed_features(device=device)
    print(f"  Loaded {data['N']:,} nodes, {data['edge_index_test'].shape[1]:,} edges, "
          f"feature dim={data['X_test'].shape[1]}")
elif RAW_CSV_PATH.exists():
    print("No pre-computed features. Preprocessing from raw CSV (slow path)...")
    data = preprocess_from_csv(RAW_CSV_PATH, device=str(device))
else:
    raise FileNotFoundError(
        f"Neither pre-computed features ({PROCESSED_DATA_DIR}) nor raw CSV ({RAW_CSV_PATH}) found."
    )

# Unpack for convenience
X_full = data["X_test"].to(device)
edge_index_full = data["edge_index_test"].to(device)
y_full = data["y"]
account_to_id = data["account_to_id"]
id_to_account = data["id_to_account"]
N = data["N"]

print(f"\nReady: {N:,} accounts | feature dim {X_full.shape[1]} | {edge_index_full.shape[1]:,} edges")

Pre-computed features found — loading (fast path)...
  Loaded 515,080 nodes, 5,078,345 edges, feature dim=58

Ready: 515,080 accounts | feature dim 58 | 5,078,345 edges


## Inference Functions

Two main entry points:
- **`run_inference_full_dataset()`** — runs the GNN on all nodes, returns a DataFrame with risk scores for every account
- **`get_laundering_prediction(account_id)`** — looks up one account from the cached inference results

In [36]:
# ============================================
# Run GNN inference on full graph
# ============================================

def _risk_label(score):
    """Map risk score to categorical label."""
    if score >= 0.5:
        return "laundering"
    elif score >= 0.2:
        return "suspicious"
    else:
        return "normal"


def run_inference_full_dataset(model, X, edge_index, id_to_account, y=None,
                                risk_threshold=DEFAULT_RISK_THRESHOLD):
    """
    Run the GNN on all nodes and return a DataFrame with columns:
        id, risk, riskScore, txCount, pattern, aiExplanation, role, cluster

    Args:
        model: trained GraphSAGE model (already on device, in eval mode)
        X: node feature tensor (N, D) on device
        edge_index: edge index tensor (2, E) on device
        id_to_account: dict mapping node_id -> account_id
        y: optional ground truth labels tensor (N,)
        risk_threshold: float, threshold for binary is_laundering flag

    Returns:
        pd.DataFrame sorted by riskScore descending, and raw risk_scores array
    """
    model.eval()
    with torch.no_grad():
        log_probs = model(X, edge_index)
    # Class 1 = laundering probability
    risk_scores = log_probs.exp()[:, 1].cpu().numpy()

    # Compute txCount per node from edge_index (out-degree + in-degree)
    num_nodes = X.shape[0]
    ei_cpu = edge_index.cpu()
    tx_count = np.zeros(num_nodes, dtype=np.int64)
    src_np = ei_cpu[0].numpy()
    dst_np = ei_cpu[1].numpy()
    np.add.at(tx_count, src_np, 1)
    np.add.at(tx_count, dst_np, 1)

    records = []
    for node_id in range(len(risk_scores)):
        acc_id = id_to_account.get(node_id, node_id)
        score = float(risk_scores[node_id])
        row = {
            "id": acc_id,
            "risk": _risk_label(score),
            "riskScore": score,
            "txCount": int(tx_count[node_id]),
            "pattern": "",
            "aiExplanation": "TOBEFILLED",
            "role": "",
            "cluster": 1,
        }
        records.append(row)

    df_results = pd.DataFrame(records).sort_values("riskScore", ascending=False).reset_index(drop=True)
    return df_results, risk_scores


# Run inference now
print("Running GNN inference on full graph...")
results_df, node_risk_scores = run_inference_full_dataset(
    model, X_full, edge_index_full, id_to_account, y=y_full
)

n_flagged = (results_df["risk"] == "laundering").sum()
print(f"Done. {N:,} accounts scored | {n_flagged:,} flagged as laundering (threshold={DEFAULT_RISK_THRESHOLD})")
print(f"\nTop 10 highest-risk accounts:")
results_df.head(10)

Running GNN inference on full graph...
Done. 515,080 accounts scored | 75,181 flagged as laundering (threshold=0.5)

Top 10 highest-risk accounts:


,id,risk,riskScore,txCount,pattern,aiExplanation,role,cluster
0,803D95360,laundering,0.999512,4,,TOBEFILLED,,1
1,804C1CD70,laundering,0.994998,5,,TOBEFILLED,,1
2,80C93DF00,laundering,0.992046,42,,TOBEFILLED,,1
3,8007C7660,laundering,0.988258,49,,TOBEFILLED,,1
4,80A9491C0,laundering,0.981813,13,,TOBEFILLED,,1
5,100428660,laundering,0.980408,169756,,TOBEFILLED,,1
6,8113EEDB0,laundering,0.979645,4,,TOBEFILLED,,1
7,80A5176A0,laundering,0.978280,5,,TOBEFILLED,,1
8,80D8F53F0,laundering,0.977857,69,,TOBEFILLED,,1
9,80B8C96C0,laundering,0.977584,13,,TOBEFILLED,,1


In [37]:
# ============================================
# Single-account lookup (uses cached full-graph inference)
# ============================================

def get_laundering_prediction(account_id, risk_threshold=None):
    """
    Look up the laundering prediction for a single account.
    Must be called AFTER run_inference_full_dataset() has populated results_df.

    Args:
        account_id: account identifier (str or int, as it appears in the dataset)
        risk_threshold: override for risk label threshold (default: DEFAULT_RISK_THRESHOLD)

    Returns:
        dict with: id, risk, riskScore, txCount, pattern, aiExplanation, role, cluster, message
    """
    if risk_threshold is None:
        risk_threshold = DEFAULT_RISK_THRESHOLD

    if node_risk_scores is None:
        return {"id": None, "risk": None, "riskScore": None, "txCount": None,
                "pattern": "", "aiExplanation": "", "role": "", "cluster": 1,
                "message": "Inference not run yet. Call run_inference_full_dataset() first."}

    # Try the key as-is, then as int, then as str
    key = account_id if account_id in account_to_id else None
    if key is None and isinstance(account_id, str) and account_id.isdigit():
        key = int(account_id)
    if key is None and isinstance(account_id, int):
        key = str(account_id)
    if key is None or key not in account_to_id:
        return {"id": account_id, "risk": None, "riskScore": None, "txCount": None,
                "pattern": "", "aiExplanation": "", "role": "", "cluster": 1,
                "message": f"Account '{account_id}' not found in graph."}

    # Look up from cached results_df
    match = results_df[results_df["id"] == key]
    if len(match) == 0:
        # Fallback: compute from raw scores
        node_id = account_to_id[key]
        score = float(node_risk_scores[node_id])
        return {
            "id": key,
            "risk": _risk_label(score),
            "riskScore": score,
            "txCount": 0,
            "pattern": "",
            "aiExplanation": "TOBEFILLED",
            "role": "",
            "cluster": 1,
            "message": "OK (fallback)",
        }

    row = match.iloc[0]
    return {
        "id": row["id"],
        "risk": row["risk"],
        "riskScore": float(row["riskScore"]),
        "txCount": int(row["txCount"]),
        "pattern": row["pattern"],
        "aiExplanation": row["aiExplanation"],
        "role": row["role"],
        "cluster": int(row["cluster"]),
        "message": "OK",
    }

print("get_laundering_prediction() ready.")

get_laundering_prediction() ready.


## Examples

Sample a few accounts and look up their predictions.

In [12]:
list(id_to_account.values())[:5]

['8000EBD30', '8000F4580', '8000F4670', '8000F5030', '8000F5200']

In [38]:
df = pd.read_csv("../100_accounts.csv")
sample_accounts = pd.unique(df[["Account","Account.1"]].astype(str).to_numpy().ravel())

In [43]:
# Look up specific accounts
new_dic = []
for acc in sample_accounts:
    r = get_laundering_prediction(acc)
    new_dic.append(r)
    print(f"Account {r['id']} -> risk={r['risk']}, riskScore={r['riskScore']:.4f}, txCount={r['txCount']}")

new_df = pd.DataFrame(new_dic)
print(f"\n--- Summary ---")
print(f"Total accounts:     {len(results_df):,}")
print(f"Laundering:         {(results_df['risk'] == 'laundering').sum():,}")
print(f"Suspicious:         {(results_df['risk'] == 'suspicious').sum():,}")
print(f"Normal:             {(results_df['risk'] == 'normal').sum():,}")

Account 100428660 -> risk=laundering, riskScore=0.9804, txCount=169756
Account 800825340 -> risk=laundering, riskScore=0.6115, txCount=97
Account 805B716C0 -> risk=laundering, riskScore=0.6135, txCount=28
Account 80B39E7B0 -> risk=laundering, riskScore=0.5886, txCount=25
Account 80DC756E0 -> risk=laundering, riskScore=0.6061, txCount=22
Account 80E480620 -> risk=laundering, riskScore=0.5753, txCount=34
Account 8140702D0 -> risk=laundering, riskScore=0.5806, txCount=29
Account 805B66DC0 -> risk=laundering, riskScore=0.5687, txCount=15
Account 805A66B50 -> risk=laundering, riskScore=0.5995, txCount=46
Account 8033AA1B0 -> risk=laundering, riskScore=0.6500, txCount=93
Account 810766370 -> risk=laundering, riskScore=0.5999, txCount=33
Account 808E5E2E0 -> risk=laundering, riskScore=0.5894, txCount=32
Account 80365D680 -> risk=laundering, riskScore=0.6024, txCount=25
Account 81108E620 -> risk=laundering, riskScore=0.5802, txCount=35
Account 806A3C0D0 -> risk=laundering, riskScore=0.6150, tx

In [44]:
sample_accounts.shape

(699,)

In [ ]:
import random

df_big = pd.read_csv(RAW_CSV_PATH)

In [97]:
# ============================================
# Build a focused subgraph for frontend visualization
# ============================================
# Strategy:
# 1. Pick 2 random laundering nodes that share an edge
# 2. Collect all their neighbors that are laundering or suspicious
# 3. From ALL collected nodes' neighbors, sample ~500 that are suspicious/normal (mostly normal)
# 4. Export nodes.csv and edges.csv for the frontend

import random

# Build a lookup: account -> risk label from results_df
risk_lookup = dict(zip(results_df["id"].astype(str), results_df["risk"]))
score_lookup = dict(zip(results_df["id"].astype(str), results_df["riskScore"]))
txcount_lookup = dict(zip(results_df["id"].astype(str), results_df["txCount"]))

# Vectorized adjacency via pandas
edges = df_big[["Account", "Account.1"]].astype(str)

# Step 1: Find two connected laundering nodes
laundering_ids = set(results_df[results_df["risk"] == "laundering"]["id"].astype(str))

laundering_edge_mask = edges["Account"].isin(laundering_ids) & edges["Account.1"].isin(laundering_ids) & (edges["Account"] != edges["Account.1"])
laundering_pairs_df = edges[laundering_edge_mask]

random.seed(42)
seed_idx = random.choice(laundering_pairs_df.index.tolist())
seed_pair = (laundering_pairs_df.at[seed_idx, "Account"], laundering_pairs_df.at[seed_idx, "Account.1"])
print(f"Seed pair: {seed_pair}")

# Step 2: From the seed pair, collect all neighbors that are laundering or suspicious
seeds = set(seed_pair)
high_risk_labels = {"laundering", "suspicious"}

# Get neighbors of seed nodes
seed_neighbors = pd.unique(
    edges[edges["Account"].isin(seeds) | edges["Account.1"].isin(seeds)][["Account", "Account.1"]].to_numpy().ravel()
)
# Keep only high-risk neighbors
high_risk_pool = set(n for n in seed_neighbors if risk_lookup.get(n) in high_risk_labels) | seeds

# Expand one more hop: neighbors of high_risk_pool that are laundering/suspicious
hop2_neighbors = pd.unique(
    edges[edges["Account"].isin(high_risk_pool) | edges["Account.1"].isin(high_risk_pool)][["Account", "Account.1"]].to_numpy().ravel()
)
high_risk_pool = set(n for n in hop2_neighbors if risk_lookup.get(n) in high_risk_labels) | high_risk_pool

print(f"High-risk nodes (laundering + suspicious neighbors): {len(high_risk_pool)}")

# Step 3: From ALL high-risk nodes' neighbors, collect suspicious/normal candidates
all_neighbors = pd.unique(
    edges[edges["Account"].isin(high_risk_pool) | edges["Account.1"].isin(high_risk_pool)][["Account", "Account.1"]].to_numpy().ravel()
)
normal_suspicious_candidates = [n for n in all_neighbors if n not in high_risk_pool and risk_lookup.get(n) in ("normal", "suspicious")]

print(f"Normal/suspicious neighbor candidates: {len(normal_suspicious_candidates)}")

# Sample ~500, weighted toward normal
random.shuffle(normal_suspicious_candidates)
suspicious_cands = [c for c in normal_suspicious_candidates if risk_lookup.get(c) == "suspicious"]
normal_cands = [c for c in normal_suspicious_candidates if risk_lookup.get(c) == "normal"]

n_target = 500
n_suspicious = min(len(suspicious_cands), n_target // 5)
n_normal = min(len(normal_cands), n_target - n_suspicious)
n_suspicious = min(len(suspicious_cands), n_target - n_normal)

sampled_low_risk = set(suspicious_cands[:n_suspicious] + normal_cands[:n_normal])
print(f"Sampled low-risk nodes: {len(sampled_low_risk)} (suspicious={n_suspicious}, normal={n_normal})")

# Final node set
all_selected = high_risk_pool | sampled_low_risk
print(f"Total selected nodes: {len(all_selected)}")

# Step 4: Build nodes DataFrame
node_records = []
for acc in all_selected:
    node_records.append({
        "id": acc,
        "risk": risk_lookup.get(acc, "normal"),
        "riskScore": score_lookup.get(acc, 0.0),
        "txCount": txcount_lookup.get(acc, 0),
        "pattern": "",
        "aiExplanation": "TOBEFILLED",
        "role": "",
        "cluster": 1,
    })
df_nodes = pd.DataFrame(node_records).sort_values("riskScore", ascending=False).reset_index(drop=True)

# Step 5: Build edges DataFrame — only edges where BOTH endpoints are in our node set (vectorized)
selected_set = set(df_nodes["id"].astype(str))
both_in = edges["Account"].isin(selected_set) & edges["Account.1"].isin(selected_set)
df_edges = pd.DataFrame({
    "source": edges.loc[both_in, "Account"].values,
    "target": edges.loc[both_in, "Account.1"].values,
    "amount": df_big.loc[both_in, "Amount Received"].values,
})

print(f"\nFinal: {len(df_nodes)} nodes, {len(df_edges)} edges")
print(f"Risk breakdown: {df_nodes['risk'].value_counts().to_dict()}")

Seed pair: ('801ED1150', '807ACA380')
High-risk nodes (laundering + suspicious neighbors): 22
Normal/suspicious neighbor candidates: 374
Sampled low-risk nodes: 374 (suspicious=42, normal=332)
Total selected nodes: 396

Final: 396 nodes, 1449 edges
Risk breakdown: {'normal': 332, 'suspicious': 49, 'laundering': 15}


In [98]:
# Save to frontend public/data/
nodes_path = PROJECT_ROOT / "frontend" / "public" / "data" / "nodes.csv"
edges_path = PROJECT_ROOT / "frontend" / "public" / "data" / "edges.csv"

df_nodes.to_csv(nodes_path, index=False)
df_edges.to_csv(edges_path, index=False)

print(f"Saved {len(df_nodes)} nodes to {nodes_path}")
print(f"Saved {len(df_edges)} edges to {edges_path}")
print(f"\nNodes sample:")
df_nodes.head(10)

Saved 396 nodes to c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\GenAI-Genesis\frontend\public\data\nodes.csv
Saved 1449 edges to c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\GenAI-Genesis\frontend\public\data\edges.csv

Nodes sample:


,id,risk,riskScore,txCount,pattern,aiExplanation,role,cluster
0,1004286A8,laundering,0.923795,103671,,TOBEFILLED,,1
1,8072CB530,laundering,0.624096,87,,TOBEFILLED,,1
2,8067B9110,laundering,0.593398,39,,TOBEFILLED,,1
3,801ED0D60,laundering,0.592330,65,,TOBEFILLED,,1
4,803154530,laundering,0.566953,54,,TOBEFILLED,,1
5,801ED1150,laundering,0.566613,16,,TOBEFILLED,,1
6,81247AEF0,laundering,0.559265,41,,TOBEFILLED,,1
7,8010D6540,laundering,0.556492,83,,TOBEFILLED,,1
8,807A84B90,laundering,0.549214,26,,TOBEFILLED,,1
9,80190B350,laundering,0.539812,92,,TOBEFILLED,,1


In [101]:
df_nodes[df_nodes["risk"] == "normal"]

,id,risk,riskScore,txCount,pattern,aiExplanation,role,cluster
64,80EC19BC0,normal,0.173273,11,,TOBEFILLED,,1
65,802617870,normal,0.158465,3,,TOBEFILLED,,1
66,8080088A0,normal,0.156868,3,,TOBEFILLED,,1
67,8001AD4E0,normal,0.153161,3,,TOBEFILLED,,1
68,80269D870,normal,0.150615,3,,TOBEFILLED,,1
...,...,...,...,...,...,...,...,...
391,80372B010,normal,0.116167,2,,TOBEFILLED,,1
392,8014C2C10,normal,0.115994,2,,TOBEFILLED,,1
393,808012B30,normal,0.115289,2,,TOBEFILLED,,1
394,80AC1D6F0,normal,0.113752,2,,TOBEFILLED,,1
